#Web Janak - Qwen 2.5 Fine-Tuning on Google Colab

This notebook fine-tunes Qwen 2.5-3B model for UI code generation using QLoRA.


---

## Setup Environment

Install required dependencies

In [6]:
%%capture
!pip install transformers>=4.43.0
!pip install accelerate>=0.30.0
!pip install peft>=0.11.0
!pip install trl>=0.9.0
!pip install datasets>=2.19.0
!pip install bitsandbytes>=0.43.0
!pip install torch>=2.1.0
!pip install scipy sentencepiece protobuf einops
!pip install -U bitsandbytes

In [7]:
# Verify accelerate is working
import accelerate
print(f"Accelerate version: {accelerate.__version__}")

import transformers
print(f"Transformers version: {transformers.__version__}")

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Accelerate version: 1.12.0
Transformers version: 4.57.3
PyTorch version: 2.9.0+cu126
CUDA available: True


## Upload Dataset

`webjanak_dataset.jsonl` using for dataset

In [8]:
# Verify dataset exists
import os
if os.path.exists('/content/webjanak_dataset.jsonl'):
    print("Dataset found!")
    with open('/content/webjanak_dataset.jsonl', 'r') as f:
        lines = f.readlines()
    print(f"Total samples: {len(lines)}")
else:
    print("Please upload webjanak_dataset.jsonl file first!")

Dataset found!
Total samples: 160


## Load Dataset & Model

Prepare the training data and load Qwen model

In [9]:
import json
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("Loading dataset...")

# Load JSONL dataset
data = []
with open('/content/webjanak_dataset.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

# Format for Qwen chat template
def format_sample(sample):
    return {
        "text": f"""<|im_start|>system
You are WebJanak AI, an expert HTML developer specialized in creating beautiful, responsive UI components following Indian government design standards. Generate complete, production-ready code.<|im_end|>
<|im_start|>user
{sample['instruction']}<|im_end|>
<|im_start|>assistant
{sample['output']}<|im_end|>"""
    }

formatted_data = [format_sample(sample) for sample in data]
dataset = Dataset.from_list(formatted_data)

print(f"Loaded {len(dataset)} samples")
print(f"\n Sample preview (first 200 chars):")
print(dataset[0]['text'][:200] + "...")

Loading dataset...
Loaded 160 samples

 Sample preview (first 200 chars):
<|im_start|>system
You are WebJanak AI, an expert HTML developer specialized in creating beautiful, responsive UI components following Indian government design standards. Generate complete, production...


In [14]:
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes


Found existing installation: bitsandbytes 0.49.1
Uninstalling bitsandbytes-0.49.1:
  Successfully uninstalled bitsandbytes-0.49.1
  Using cached bitsandbytes-0.49.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.1-py3-none-manylinux_2_24_x86_64.whl (59.1 MB)


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading model: {MODEL_NAME}")

try:
    # Quantization config for memory efficiency
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    print("\nStep 1/2: Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        use_fast=False
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    print(f"Tokenizer loaded (vocab size: {len(tokenizer)})")

    print("\nStep 2/2: Loading model with 4-bit quantization...")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )

    device = next(model.parameters()).device
    print(f"\nModel loaded successfully!")
    print(f"Model device: {device}")

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        print(f"GPU Memory: {allocated:.2f}GB allocated")

except Exception as e:
    print(f"\nError: {str(e)}")
    raise

Loading model: Qwen/Qwen2.5-3B-Instruct

Step 1/2: Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer loaded (vocab size: 151665)

Step 2/2: Loading model with 4-bit quantization...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully!
Model device: cuda:0
GPU Memory: 1.91GB allocated


## Configure LoRA

Setup QLoRA for efficient fine-tuning

In [2]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# LoRA configuration
lora_config = LoraConfig(
    r=8,                    # Rank
    lora_alpha=16,          # Scaling factor
    target_modules=[       # Which layers to adapt
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Statistics:")
print(f"Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total params: {total_params:,}")
print(f"\nLoRA configured!")


Model Statistics:
Trainable params: 3,686,400 (0.22%)
Total params: 1,702,359,040

LoRA configured!


##Train the Model!


In [3]:
from datasets import load_dataset

# Load JSONL dataset
dataset = load_dataset(
    "json",
    data_files="/content/webjanak_dataset.jsonl",
    split="train"
)

print(f"Dataset loaded with {len(dataset)} samples")
print("Columns:", dataset.column_names)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset loaded with 160 samples
Columns: ['instruction', 'output', 'category', 'style_variant']


In [4]:
def tokenize_function(examples):
    texts = []

    for inst, out in zip(examples["instruction"], examples["output"]):
        text = f"""### Instruction:
{inst}

### Response:
{out}"""
        texts.append(text)

    result = tokenizer(
        texts,
        truncation=True,
        max_length=1024,
        padding=False,
    )

    result["labels"] = result["input_ids"].copy()
    return result


In [5]:
from transformers import Trainer, TrainingArguments
import torch
import gc

# ----------------------------
# CLEAN MEMORY
# ----------------------------
torch.cuda.empty_cache()
gc.collect()

# ----------------------------
# PAD TOKEN FIX (Qwen needs this)
# ----------------------------
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# PROMPT TEMPLATE
# ----------------------------
def format_prompt(example):
    return f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""

# ----------------------------
# TOKENIZATION FUNCTION
# ----------------------------
def tokenize_function(examples):
    texts = [
        format_prompt(
            {
                "instruction": inst,
                "output": out
            }
        )
        for inst, out in zip(examples["instruction"], examples["output"])
    ]

    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=1024,
        padding=False,
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("🔄 Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
)
print(f"✅ Tokenized {len(tokenized_dataset)} samples\n")

# ----------------------------
# CUSTOM DATA COLLATOR
# ----------------------------
class CustomDataCollator:
    def __init__(self, tokenizer, max_length=1024):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        max_len = min(
            max(len(f["input_ids"]) for f in features),
            self.max_length
        )

        batch = {
            "input_ids": [],
            "attention_mask": [],
            "labels": []
        }

        for f in features:
            input_ids = f["input_ids"][:max_len]
            attention_mask = f["attention_mask"][:max_len]
            labels = f["labels"][:max_len]

            pad_len = max_len - len(input_ids)

            batch["input_ids"].append(
                input_ids + [self.tokenizer.pad_token_id] * pad_len
            )
            batch["attention_mask"].append(
                attention_mask + [0] * pad_len
            )
            batch["labels"].append(
                labels + [-100] * pad_len
            )

        return {
            "input_ids": torch.tensor(batch["input_ids"]),
            "attention_mask": torch.tensor(batch["attention_mask"]),
            "labels": torch.tensor(batch["labels"]),
        }

data_collator = CustomDataCollator(tokenizer)

# ----------------------------
# TRAINING ARGUMENTS (T4 SAFE)
# ----------------------------
training_args = TrainingArguments(
    output_dir="./webjanak-qwen-finetuned",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    warmup_steps=50,
    logging_steps=5,
    save_steps=50,
    save_total_limit=1,
    fp16=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    group_by_length=True,
    report_to="none",
    load_best_model_at_end=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_steps=200,
)

# ----------------------------
# TRAINER
# ----------------------------
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator,
)

# ----------------------------
# GPU CHECK
# ----------------------------
if torch.cuda.is_available():
    print("GPU Memory before training:")
    print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB\n")

# ----------------------------
# TRAIN
# ----------------------------
print("=" * 60)
print("🔥 STARTING TRAINING")
print("=" * 60)

try:
    trainer.train()
    print("\nTRAINING COMPLETE!")

except RuntimeError as e:
    if "out of memory" in str(e):
        print("\nCUDA OOM")
        print("Try reducing max_length to 512")
    raise


🔄 Tokenizing dataset...


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

✅ Tokenized 160 samples

GPU Memory before training:
Allocated: 2.51 GB
Reserved: 4.11 GB

🔥 STARTING TRAINING


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,1.025400
10,0.967400
15,0.960900
20,0.818200
25,0.645100
30,0.494700
35,0.403600
40,0.342700
45,0.293500
50,0.232900


Step,Training Loss
5,1.025400
10,0.967400
15,0.960900
20,0.818200
25,0.645100
30,0.494700
35,0.403600
40,0.342700
45,0.293500
50,0.232900



TRAINING COMPLETE!


In [6]:
from datasets import load_dataset

DATA_PATH = "/content/webjanak_dataset.jsonl"

raw_dataset = load_dataset(
    "json",
    data_files=DATA_PATH,
    split="train"
)

print(f"Loaded {len(raw_dataset)} samples")
print(raw_dataset[0])


Loaded 160 samples
{'instruction': 'Create a modern portfolio website with hero section, project gallery, skills grid, and contact form', 'output': '<!DOCTYPE html>\n<html><head><meta charset="UTF-8"><title>Portfolio</title>\n<style>\n* { margin: 0; padding: 0; box-sizing: border-box; }\nbody { font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, Oxygen, Ubuntu, sans-serif; background: #ffffff; }\nnav { position: fixed; top: 0; width: 100%; background: white; padding: 1.5rem 5%; box-shadow: 0 2px 10px rgba(0,0,0,0.05); z-index: 100; }\nnav ul { display: flex; list-style: none; gap: 2rem; }\nnav a { text-decoration: none; color: #333; font-weight: 500; transition: color 0.3s; }\nnav a:hover { color: #fa709a; }\n.hero { min-height: 100vh; display: flex; align-items: center; padding: 0 10%; margin-top: 80px; }\n.hero-text h1 { font-size: 5rem; font-weight: 900; color: #111; line-height: 1.1; margin-bottom: 1rem; }\n.hero-text .highlight { color: #fa709a; }\n.hero-text p {

In [7]:
SYSTEM_PROMPT = (
    "You are WebJanak AI, an expert React and HTML developer. "
    "Generate clean, modern, responsive UI code."
)

def to_qwen_chat(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['instruction']}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['output']}\n"
        "<|im_end|>"
    )
    return {"text": text}

formatted_dataset = raw_dataset.map(
    to_qwen_chat,
    remove_columns=raw_dataset.column_names
)

print("\n✅ Qwen-formatted sample:\n")
print(formatted_dataset[0]["text"][:600])


Map:   0%|          | 0/160 [00:00<?, ? examples/s]


✅ Qwen-formatted sample:

<|im_start|>system
You are WebJanak AI, an expert React and HTML developer. Generate clean, modern, responsive UI code.
<|im_end|>
<|im_start|>user
Create a modern portfolio website with hero section, project gallery, skills grid, and contact form
<|im_end|>
<|im_start|>assistant
<!DOCTYPE html>
<html><head><meta charset="UTF-8"><title>Portfolio</title>
<style>
* { margin: 0; padding: 0; box-sizing: border-box; }
body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, sans-serif; background: #ffffff; }
nav { position: fixed; top: 0; width: 100%; background: w


In [8]:
def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024,
        padding=False,
    )
    result["labels"] = [ids[:] for ids in result["input_ids"]]
    return result

print("\nTokenizing dataset...")
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_dataset.column_names,
)

print(f"Tokenized {len(tokenized_dataset)} samples")



Tokenizing dataset...


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Tokenized 160 samples


In [10]:
from datasets import load_dataset

# Load dataset
raw_dataset = load_dataset(
    "json",
    data_files="/content/webjanak_dataset.jsonl",
    split="train"
)

print("Raw sample:")
print(raw_dataset[0])

SYSTEM_PROMPT = (
    "You are WebJanak AI, an expert React and HTML developer. "
    "Generate clean, modern, responsive UI code."
)

def format_qwen_chat(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['instruction']}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['output']}\n"
        "<|im_end|>"
    )
    return {"text": text}

formatted_dataset = raw_dataset.map(
    format_qwen_chat,
    remove_columns=raw_dataset.column_names
)

print("\nFormatted sample:")
print(formatted_dataset[0]["text"][:500])


Raw sample:
{'instruction': 'Create a modern portfolio website with hero section, project gallery, skills grid, and contact form', 'output': '<!DOCTYPE html>\n<html><head><meta charset="UTF-8"><title>Portfolio</title>\n<style>\n* { margin: 0; padding: 0; box-sizing: border-box; }\nbody { font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, Oxygen, Ubuntu, sans-serif; background: #ffffff; }\nnav { position: fixed; top: 0; width: 100%; background: white; padding: 1.5rem 5%; box-shadow: 0 2px 10px rgba(0,0,0,0.05); z-index: 100; }\nnav ul { display: flex; list-style: none; gap: 2rem; }\nnav a { text-decoration: none; color: #333; font-weight: 500; transition: color 0.3s; }\nnav a:hover { color: #fa709a; }\n.hero { min-height: 100vh; display: flex; align-items: center; padding: 0 10%; margin-top: 80px; }\n.hero-text h1 { font-size: 5rem; font-weight: 900; color: #111; line-height: 1.1; margin-bottom: 1rem; }\n.hero-text .highlight { color: #fa709a; }\n.hero-text p { font-s

Map:   0%|          | 0/160 [00:00<?, ? examples/s]


Formatted sample:
<|im_start|>system
You are WebJanak AI, an expert React and HTML developer. Generate clean, modern, responsive UI code.
<|im_end|>
<|im_start|>user
Create a modern portfolio website with hero section, project gallery, skills grid, and contact form
<|im_end|>
<|im_start|>assistant
<!DOCTYPE html>
<html><head><meta charset="UTF-8"><title>Portfolio</title>
<style>
* { margin: 0; padding: 0; box-sizing: border-box; }
body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, 


## 💾 Step 6: Save Model

Save the fine-tuned model

In [11]:
# Save model and tokenizer
output_dir = "./webjanak-qwen-finetuned"

print("💾 Saving model...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model saved to: {output_dir}")
print("\n📥 Download options:")
print("   1. Click folder icon → Right-click folder → Download")
print("   2. Save to Google Drive (see next cell)")

💾 Saving model...
✅ Model saved to: ./webjanak-qwen-finetuned

📥 Download options:
   1. Click folder icon → Right-click folder → Download
   2. Save to Google Drive (see next cell)


In [12]:
# Optional: Save to Google Drive
from google.colab import drive
import shutil

drive.mount('/content/drive')

# Copy to Drive
drive_path = '/content/drive/MyDrive/webjanak-qwen-finetuned'
shutil.copytree(output_dir, drive_path, dirs_exist_ok=True)

print(f"✅ Model saved to Google Drive: {drive_path}")

Mounted at /content/drive
✅ Model saved to Google Drive: /content/drive/MyDrive/webjanak-qwen-finetuned


## 🧪 Step 7: Test the Model

Try generating some UI code!

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

print("📂 Loading fine-tuned model...")

# Base model name
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
FINETUNED_PATH = "./webjanak-qwen-finetuned"

# Load tokenizer
print("📝 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("✅ Tokenizer loaded")

# Load base model
print("\n🤖 Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

# Load LoRA adapters
print("🔧 Loading LoRA adapters...")
model = PeftModel.from_pretrained(model, FINETUNED_PATH)
model = model.merge_and_unload()  # Merge adapters for faster inference

print("\n✅ Fine-tuned model loaded successfully!")

# Check device
device = next(model.parameters()).device
print(f"📊 Model device: {device}")

if torch.cuda.is_available():
    print(f"🎮 GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB\n")

📂 Loading fine-tuned model...
📝 Loading tokenizer...
✅ Tokenizer loaded

🤖 Loading base model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔧 Loading LoRA adapters...

✅ Fine-tuned model loaded successfully!
📊 Model device: cuda:0
🎮 GPU Memory: 8.29GB



In [14]:
import os

# Check what files exist
FINETUNED_PATH = "./webjanak-qwen-finetuned"

print("📂 Checking saved files...")
if os.path.exists(FINETUNED_PATH):
    files = os.listdir(FINETUNED_PATH)
    print(f"\n✅ Directory exists with {len(files)} files:")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(FINETUNED_PATH, f)) / (1024*1024)
        print(f"   - {f} ({size:.2f} MB)")
else:
    print(f"❌ Directory '{FINETUNED_PATH}' does not exist!")

📂 Checking saved files...

✅ Directory exists with 11 files:
   - README.md (0.00 MB)
   - adapter_config.json (0.00 MB)
   - adapter_model.safetensors (14.10 MB)
   - added_tokens.json (0.00 MB)
   - chat_template.jinja (0.00 MB)
   - checkpoint-200 (0.00 MB)
   - merges.txt (1.59 MB)
   - special_tokens_map.json (0.00 MB)
   - tokenizer_config.json (0.00 MB)
   - training_args.bin (0.01 MB)
   - vocab.json (3.23 MB)


In [18]:
# ============================================================
# WebJanak AI – (Base Model + LoRA)
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# -------------------------------
# CONFIG
# -------------------------------
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LORA_PATH = "/content/webjanak-qwen-finetuned"

MAX_NEW_TOKENS = 1024
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# LOAD TOKENIZER (BASE MODEL ONLY)
# -------------------------------
print("📝 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_fast=False
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------
# LOAD BASE MODEL (4-bit)
# -------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("🤖 Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},   # 👈 FORCE ALL MODULES TO GPU
    trust_remote_code=True
)

# -------------------------------
# ATTACH LORA
# -------------------------------
print("🔗 Attaching LoRA adapter...")
model = PeftModel.from_pretrained(
    model,
    LORA_PATH,
    device_map="auto"
)

model.eval()
print("✅ Model & LoRA loaded successfully!\n")

# ============================================================
# GENERATION FUNCTION
# ============================================================
def generate_ui(prompt, max_tokens=MAX_NEW_TOKENS):

    messages = [
        {
            "role": "system",
            "content": (
                "You are WebJanak AI, an expert React and HTML developer. "
                "Generate clean, modern, responsive UI code. "
                "Prefer semantic HTML and Tailwind-style layouts when possible."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Clean assistant output
    if "<|im_start|>assistant" in decoded:
        decoded = decoded.split("<|im_start|>assistant")[-1]

    return decoded.strip()

# ============================================================
# TEST PROMPTS
# ============================================================
test_prompts = [
    "Create a modern portfolio website with hero section",
    "Build a coffee shop landing page with menu",
    "Design a responsive admin dashboard with stats cards"
]

print("🧪 Testing WebJanak AI...\n")

for i, prompt in enumerate(test_prompts, 1):
    print(f"🔹 Test {i}/{len(test_prompts)}")
    print(f"🧠 Prompt: {prompt}")
    print("-" * 60)

    output = generate_ui(prompt)

    print(f"📏 Output Length: {len(output)} characters")
    print("\n📄 Preview (first 300 chars):\n")
    print(output[:300])
    print("\n" + "=" * 60 + "\n")

print("🎉 All tests completed successfully!")


📝 Loading tokenizer...
🤖 Loading base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 594.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 124.12 MiB is free. Process 40324 has 14.62 GiB memory in use. Of the allocated memory 14.36 GiB is allocated by PyTorch, and 124.81 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 📊 Step 8: Training Summary

View final statistics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print("="*60)
print("📊 FINE-TUNING SUMMARY")
print("="*60)
print(f"\n📁 Dataset: {len(dataset)} samples")
print(f"🤖 Model: {MODEL_NAME}")
print(f"⚡ Method: QLoRA (4-bit)")
print(f"📈 Epochs: {training_args.num_train_epochs}")
print(f"💾 Output: {output_dir}")
print(f"\n✅ Model is ready for deployment!")
print("\n🚀 Next steps:")
print("   1. Download model from Colab")
print("   2. Copy to project: d:/text to react ui/qwen-finetuning/")
print("   3. Update .env: USE_QWEN=true")
print("   4. Restart server and test!")